# Template Task: Multi-Regime Thermal Load Regression

This notebook is the interactive companion for `tasks/template/`. Both this
notebook and `run_template.py` import their shared task configuration and
student instructions from `config.py`. Copy `tasks/template/` to
`tasks/<your_task>/` and update the `# CUSTOMIZE:` placeholders in
`data_loader.py` and `config.py`. Clear all cell outputs before committing.

### Running in Colab
If you are running this notebook in Google Colab:
1. **API Key**: Save your `GEMINI_API_KEY` under Colab **Secrets** (key icon 🔑 on the left sidebar) with notebook access enabled.
2. **Install**: Run `!pip install "codegen-classroom[notebook]"` (or `[all]`).

## Setup


In [ ]:
from collections import defaultdict
import functools
import os
import tempfile

from codegen_classroom import agent_config
from codegen_classroom import agents
from codegen_classroom import data_utils
from codegen_classroom import plotting
from codegen_classroom import reporting
from codegen_classroom import scoring
from codegen_classroom import student_learning
from codegen_classroom.tasks.template import config
from codegen_classroom.tasks.template import data_loader
from codegen_classroom.tasks.template import run_template
import pandas as pd
from sklearn.model_selection import train_test_split

agent_config.load_env()
reporting.suppress_common_warnings()
DARK_MODE = False

## Data


In [ ]:
TASK_DIR = run_template.resolve_task_dir()
DATA_DIR = os.path.join(TASK_DIR, "data")
LEADERBOARD_PATH = os.path.join(TASK_DIR, "leaderboard.tsv")

if os.path.exists(LEADERBOARD_PATH):
  reporting.render_leaderboard(LEADERBOARD_PATH)

X, y = data_loader.load_data(DATA_DIR)

X_learning, X_test_final, y_learning, y_test_final = train_test_split(
    X, y, test_size=0.2, random_state=config.VAL_SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_learning, y_learning, test_size=0.2, random_state=config.VAL_SEED
)
print(f"train={len(X_train)} val={len(X_val)} test={len(X_test_final)}")

## Student Instructions

`config.build_student_instructions` formats `config.STUDENT_INSTRUCTIONS_TEMPLATE`
with the dataset dimensions, metric weights, and runtime budget.

In [ ]:
METRIC_WEIGHTS = dict(config.DEFAULT_METRIC_WEIGHTS)
STUDENT_FUNCTION_NAME = config.STUDENT_FUNCTION_NAME

STUDENT_INSTRUCTIONS = config.build_student_instructions(
    feature_names=list(X.columns),
    target_column=y.columns[0],
    train_rows=len(X_train),
    val_rows=len(X_val),
    final_eval_rows=len(X_learning),
    timeout_s=config.RUN_DEFAULTS["timeout_s"],
    final_eval_timeout_s=config.RUN_DEFAULTS["final_eval_timeout_s"],
    metric_weights=METRIC_WEIGHTS,
)
STUDENT_INITIAL_PROMPT = config.STUDENT_INITIAL_PROMPT
STUDENT_REFINE_OPTIONS = list(config.STUDENT_REFINE_INSTRUCTION_OPTIONS)

reporting.render_markdown(STUDENT_INSTRUCTIONS)

## Classroom Setup

Run outputs are routed to `CODEGEN_EXPERIMENTS_DIR` (or the system temp directory)
so intermediate checkpoints do not clutter the repository.

In [ ]:
# @title Classroom Setup
EXPERIMENT_NAME = "template_notebook"  # @param {type:"string"}
NUM_STUDENTS = config.RUN_DEFAULTS["num_students"]  # @param {type:"integer"}
NUM_LEARNING_CYCLES = config.RUN_DEFAULTS["num_cycles"]  # @param {type:"integer"}
MODEL_NAMES = [
    config.DEFAULT_STUDENT_MODELS[i % len(config.DEFAULT_STUDENT_MODELS)]
    for i in range(NUM_STUDENTS)
]

RUNS_BASE_PATH = os.environ.get(
    "CODEGEN_EXPERIMENTS_DIR",
    os.path.join(tempfile.gettempdir(), "codegen_classroom", config.TASK_NAME),
)
RUN_ID = data_utils.generate_run_id()
RUN_DIR = os.path.join(RUNS_BASE_PATH, EXPERIMENT_NAME, RUN_ID)
data_utils.create_dir(RUN_DIR)

logger = reporting.setup_logger(
    "codegen_classroom", log_file=os.path.join(RUN_DIR, "run.log")
)

student_cfgs = student_learning.get_classroom_student_configs(
    system_instruction=STUDENT_INSTRUCTIONS,
    model_names=MODEL_NAMES,
)
students = {
    cfg.name: agents.StudentAgent(config=cfg, run_base_dir=RUN_DIR)
    for cfg in student_cfgs
}


## Learning


In [ ]:
current_classroom = list(students.values())
results_history = defaultdict(list)
active_metrics = ["overall_score"] + list(METRIC_WEIGHTS.keys())
validator = functools.partial(scoring.prediction_validator, expected_columns=1)

for current_cycle in range(NUM_LEARNING_CYCLES):
  if not current_classroom:
    break
  classroom_feedback = scoring.get_classroom_feedback_string(
      cycle_num=current_cycle,
      current_classroom=current_classroom,
      classroom_results_history=results_history,
      metric_keys=active_metrics,
  )
  classroom_results = student_learning.run_parallel_classroom_learning_cycle(
      current_classroom=current_classroom,
      x_train=X_train,
      y_train=y_train,
      x_val=X_val,
      y_val=y_val,
      current_cycle=current_cycle,
      student_initial_prompt=STUDENT_INITIAL_PROMPT,
      student_function_name=STUDENT_FUNCTION_NAME,
      code_execution_max_retries=2,
      code_execution_timeout_seconds=config.RUN_DEFAULTS["timeout_s"],
      metric_weights=METRIC_WEIGHTS,
      early_stopping_patience=config.RUN_DEFAULTS["patience"],
      student_refine_instruction_options=STUDENT_REFINE_OPTIONS,
      max_mutation_count=config.RUN_DEFAULTS["max_mutations"],
      mutation_strategy_options=[
          agents.MutationStrategy.SELF_BEST_CODE,
          agents.MutationStrategy.BEST_PEER_CROSSOVER,
          agents.MutationStrategy.CONTRASTIVE_PEER_CROSSOVER,
      ],
      validator=validator,
      classroom_feedback=classroom_feedback,
      num_workers=len(current_classroom),
  )
  current_classroom = student_learning.process_cycle_results(
      current_classroom, classroom_results, results_history, current_cycle
  )
  for student in students.values():
    student.reset_history()

plotting.plot_progress(
    os.path.join(RUN_DIR, "students"),
    metric_names=active_metrics,
    dark_mode=DARK_MODE,
)

## Evaluation


In [ ]:
FINAL_RESULTS_PATH = os.path.join(RUN_DIR, "final_eval_results.json")

eval_results, _ = student_learning.run_parallel_classroom_final_evaluation(
    students=students,
    x_learning=X_learning,
    y_learning=y_learning,
    x_test_final=X_test_final,
    y_test_final=y_test_final,
    student_function_name=STUDENT_FUNCTION_NAME,
    metric_weights=METRIC_WEIGHTS,
    code_execution_timeout_seconds=config.RUN_DEFAULTS["final_eval_timeout_s"],
    enforce_timeout=True,
    strategies=["first", "best", "last"],
    validator=validator,
    num_workers=len(students),
    final_results_path=FINAL_RESULTS_PATH,
    predictions_dir=os.path.join(RUN_DIR, "predictions"),
)

plotting.plot_final_test(
    metrics=active_metrics,
    final_results_path=FINAL_RESULTS_PATH,
    dark_mode=DARK_MODE,
)

sections = reporting.build_run_report(
    RUN_DIR, students=list(students.values()), metric_weights=METRIC_WEIGHTS
)
reporting.render_report(sections, dark_mode=DARK_MODE)
reporting.write_report(RUN_DIR, sections)

## Code Analysis


In [ ]:
# @title Best Solution

# Find the winner from evaluation results
best_student_name, best_eval_data = (
    student_learning.get_best_solution_from_eval_results(
        eval_results,
        metric_key="overall_score",
        solution_strategy="best",
        min_cycle=0,
    )
)

if best_student_name:
  student_csv_path = os.path.join(RUN_DIR, "students", f"{best_student_name}.csv")
  if os.path.exists(student_csv_path):
    student_history_df = pd.read_csv(student_csv_path)
    top_cycle_result = student_history_df.sort_values(
        by="overall_score", ascending=False
    ).iloc[0]

    reporting.render_markdown(
        (
            "# Winner: %s\n### Score: %0.4f (Cycle: %d)\n### Plan\n%s\n\n###"
            " Code\n```python\n%s\n```\n\n### Execution Output\n```\n%s\n```"
        )
        % (
            best_student_name,
            top_cycle_result["overall_score"],
            top_cycle_result["cycle"],
            top_cycle_result["plan"],
            top_cycle_result["code"],
            top_cycle_result["execution_feedback"],
        )
    )